# 04 — Modeling
Builds, trains, and saves a 1D CNN model for multi-label ECG classification on PTB-XL.
Loads pre-processed data from `data/processed/`.

In [ ]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os, pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device    : {DEVICE}")

In [ ]:
# ============================================================
# CELL 2: LOAD FROM data/processed/
# ============================================================
SAVE_DIR  = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'data', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

X = np.load(os.path.join(SAVE_DIR, 'X_clean.npy'))   # (N, 1000, 12)
y = np.load(os.path.join(SAVE_DIR, 'y.npy'))          # (N, 5)

with open(os.path.join(SAVE_DIR, 'classes.pkl'), 'rb') as f:
    CLASSES = pickle.load(f)

print(f"X: {X.shape}  y: {y.shape}")
print(f"Classes: {CLASSES}")

In [ ]:
# ============================================================
# CELL 3: PREPARE TENSORS & DATALOADERS
# PyTorch expects (N, channels, length) — we transpose leads to channels
# ============================================================

X_t = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)  # (N, 12, 1000)
y_t = torch.tensor(y, dtype=torch.float32)

X_train, X_val, y_train, y_val = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42
)

train_ds = TensorDataset(X_train, y_train)
val_ds   = TensorDataset(X_val,   y_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ============================================================
# CELL 4: DEFINE 1D CNN MODEL
# ============================================================

class ECG_CNN(nn.Module):
    def __init__(self, n_leads=12, n_classes=5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(n_leads, 32, kernel_size=7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),      nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),     nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.encoder(x))

model = ECG_CNN(n_leads=12, n_classes=len(CLASSES)).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready ✅  Trainable params: {total_params:,}")
print(model)

In [ ]:
# ============================================================
# CELL 5: TRAIN
# ============================================================

criterion = nn.BCEWithLogitsLoss()   # multi-label loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────
    model.train()
    train_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # ── Validate ───────────────────────────────────────────────
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            val_loss += criterion(model(xb), yb).item()

    tl = train_loss / len(train_loader)
    vl = val_loss  / len(val_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    print(f"Epoch {epoch:>2}/{EPOCHS}  train_loss={tl:.4f}  val_loss={vl:.4f}")

print("\n✅ Training complete!")

In [ ]:
# ============================================================
# CELL 6: PLOT TRAINING CURVES
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(history['train_loss'], label='Train Loss', color='#6366f1')
plt.plot(history['val_loss'],   label='Val Loss',   color='#f59e0b')
plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
plt.title('Training vs Validation Loss', fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 7: SAVE MODEL → data/models/ecg_cnn.pt
# ============================================================

model_path = os.path.join(MODEL_DIR, 'ecg_cnn.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'classes':          CLASSES,
    'history':          history,
}, model_path)

print(f"Model saved → {model_path} ✅")
print("✅ Ready for 05_evaluation.ipynb")